In [13]:
from MambaClassificationModel import MambaClassificationModel, HARMambaConfig
from data.OPPORTUNITY_data import load_OPP_loco_data, data_split_OPP, make_loaders_OPP
from test_mamba import test_model, save_json 
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from mamba_ssm.modules.mamba2 import Mamba2
import mamba_ssm.modules.mamba2 as mm
from data.preprocessing import fit_labelencoder, Dataset_HAR
from datetime import datetime
import torch.nn.functional as F
import mamba_ssm
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from utils import set_seed
import argparse
from torch.profiler import profile, ProfilerActivity
import time
try:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn
except ImportError:
    RMSNorm, layer_norm_fn, rms_norm_fn = None, None, None
from collections import Counter

In [12]:
ls

MambaClassificationModel.py  TEST_MAIN_SP.ipynb  test_stratify.ipynb
Mamba_SUP_AdamW/             __init__.py         train_sup_mamba.py
Mamba_blocks.py              __pycache__/        train_sup_mamba.slurm
RUN_1_WCE/                   data/               training_curves.ipynb
RUN_2_CE/                    test_mamba.py       utils.py


In [14]:
import os
base = os.path.dirname(mamba_ssm.__file__)
print(base)

/home/kmercad/myenv/lib/python3.10/site-packages/mamba_ssm


In [15]:
print(f"Mamba version: {mamba_ssm.__version__}")
print(f"Path: {mm.__file__}")

Mamba version: 2.3.0
Path: /home/kmercad/myenv/lib/python3.10/site-packages/mamba_ssm/modules/mamba2.py


In [16]:
import sys
print(sys.executable)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


/home/kmercad/myenv/bin/python
2.5.1+cu121
True
1
NVIDIA A100-SXM4-80GB


In [17]:
training_files, validation_files, test_files = data_split_OPP(4)
X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows = load_OPP_loco_data(training_files, validation_files, test_files, verbose = False)

--------------------STRATIY VAL/TEST--------------------
Validation GROUPS
group_id
S2-ADL5    22969
S3-ADL5    20648
S4-ADL1    28905
S4-ADL2    19679
Name: count, dtype: int64
Test GROUPS
group_id
S1-ADL5    22440
S4-ADL3    17621
S4-ADL4    15060
S4-ADL5    23817
Name: count, dtype: int64 


 Validation Split Label Proportion
45
1    0.482468
2    0.242416
4    0.219412
5    0.055704
Name: proportion, dtype: float64
Test Split Label Proportion
45
1    0.443652
2    0.293040
4    0.212014
5    0.051293
Name: proportion, dtype: float64


In [21]:
def label_counts(y_windows, title):
    y_windows = [int(window) for window in y_windows]
    print(f"{title}: {sorted(Counter(y_windows).items())}")
label_counts(y_windows, "Label count train")
label_counts(y_validation_windows, "Label count val")

label_counts(y_test_windows, "Label count test")

Label count train: [(1, 4652), (2, 2791), (4, 2704), (5, 528)]
Label count val: [(1, 1452), (2, 766), (4, 675), (5, 170)]
Label count test: [(1, 1139), (2, 786), (4, 560), (5, 137)]


In [8]:
#Generated from diff model
X_synthetic_all = np.load("X_synthetic_OPP1_cos.npy")
y_synthetic_all = np.load("y_synthetic_OPP1_cos.npy")

# Concatenate with real data

X_augmented = np.concatenate([X_windows, X_synthetic_all], axis=0)
y_augmented = np.concatenate([y_windows, y_synthetic_all], axis=0)

In [9]:
print("Original training set:", X_windows.shape)
print("Augmented training set:", X_augmented.shape)

Original training set: (10699, 150, 45)
Augmented training set: (11327, 150, 45)


In [22]:
@torch.no_grad()
def validate_model(model, val_loader, device, criterion):
    '''
    Validation: avg loss per window and accuracy per window.
    '''
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    for x_batch, y_batch in val_loader:
        x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)
        logits_ = model(x_batch)                                               # forward Pass
        loss = criterion(logits_, y_batch)                                     # computes mean batch loss

        bsize = y_batch.size(0)
        total_loss += loss.item() * bsize                                      # Total loss contribution of this batch

        predictions = logits_.argmax(dim = -1)                                 # gets the predicted class for each window
        total_correct += (predictions == y_batch).sum().item()                 # counts correct predictions
        total_samples += bsize

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    report = classification_report(all_labels, all_preds, target_names = ["STAND", "WALK", "SIT", "LIE"])
    return total_loss / total_samples, total_correct / total_samples, report


In [23]:
# Inference

# Load model for test
def load_model_for_inference(config, num_classes, dataset, fold, seed) -> MambaClassificationModel:
    model = MambaClassificationModel(config, num_classes).to(device, non_blocking = True)
    model.load_state_dict(torch.load(f"model_pt/model_{str(dataset)}_fold{str(fold)}_seed{seed}.pt", map_location = device))
    return model
    
@torch.no_grad()
def test_model(model, test_loader, device):
    '''
    Testing model
    '''
    model.eval()
    all_preds = []
    all_labels = []
    for x_batch, y_batch in test_loader:
        x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)
        
        logits_ = model(x_batch)
        predictions = logits_.argmax(dim = -1)

        all_preds.extend(predictions.numpy(force = True)) # move to CPU
        all_labels.extend(y_batch.numpy(force = True)) # move to CPU

    acc = accuracy_score(all_labels, all_preds)    
    report = classification_report(all_labels, all_preds, target_names = ["STAND", "WALK", "SIT", "LIE"])
    f1 = f1_score(all_labels, all_preds, average = "macro") # macro: F1 avg equally among classes, for general performance
    conf_matrix = confusion_matrix(all_labels, all_preds)
    return acc, report, f1, conf_matrix

In [37]:
for seed in [42, 58, 7, 128, 92]: # [42, 58, 7, 128, 92]
    g = set_seed(seed)
    train_loader, val_loader, test_loader, label_encoder = make_loaders_OPP(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator = g, verbose = True)
    
    #  ----------- TRAINING SETUP -----------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = int(len(label_encoder.classes_))
    config = HARMambaConfig()
    model = MambaClassificationModel(config, num_classes = num_classes)
    model.to(device, non_blocking = True) 

    
    with open("optimizer_assignment_table.txt", "w", encoding="utf-8") as f:
        print("-" * 95, file=f)
        print("Parameter assignment for the hybrid Muon + AdamW optimizer", file=f)
        print("-" * 95, file=f)
        print(f"{'NAME':<60} {'SHAPE':<25} {'OPTIMIZER CANDIDATE'}", file=f)
        print("-" * 95, file=f)
    
        for name, param in model.named_parameters():
            if param.ndim == 2:
                opt = "MUON"
            else:
                opt = "AdamW"
    
            print(f"{name:<60} {str(list(param.shape)):<25} {opt}", file=f)
    
    print("Saved as optimizer_assignment_table.txt")

      
    # -- Reset Peak Stats --
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats() 
    # ----------------------
    num_epochs = 30
    lr = 0.0006
    patience = 4
    #WCE
    label_encoder = fit_labelencoder(X_windows, y_windows)
    y_train_encoded = label_encoder.transform(np.asarray(y_windows))
    counts = np.bincount(y_train_encoded, minlength = num_classes)
    weights = torch.tensor(1.0 / counts, dtype=torch.float32, device = device)
    criterion = nn.CrossEntropyLoss(weight = weights)
    #criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = 1e-4)
    
    #  ----------- TRAINING -----------
    model_name = f"models_pt/model_OPP_fold{4}_seed{seed}.pt"
    epoch_history = []
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = None
    best_state = None
    bad_epochs = 0
    prof_out = None
    with open(f"logs/training_OPP_fold{4}.txt", "a") as log_file:
        log_file.write(f"\nTRAINING STARTING AT: {datetime.now()}\n")
        log_file.write(f"Model: {model_name} | SEED: {seed}\n")
        log_file.flush()
        for epoch in range(num_epochs):
            epoch_start = time.time() # START EPOCH TIME
            model.train()
            total_loss = 0.0
            total_correct = 0
            total_samples = 0
    
            loop = tqdm(train_loader, desc= f"Epoch {epoch+1}/{num_epochs}")
            for batch_idx, (x_batch, y_batch) in enumerate(loop):
                x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)    
                #################### PROFILING ####################
                if device.type == "cuda" and epoch == 0 and batch_idx == 1:
                    with profile(
                        activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA],
                        record_shapes = True,
                        profile_memory = True
                    ) as prof_:
                        optimizer.zero_grad()                                             # clear previous gradients
                        logits_ = model(x_batch)                                          # forward pass
                        loss = criterion(logits_, y_batch)                                # computes mean batch loss
                        loss.backward()                                                   # backward pass: compute gradients
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
                        optimizer.step()
                        torch.cuda.synchronize()
                    prof_out = prof_.key_averages().table(sort_by = "self_cuda_time_total", row_limit = 10)
                else:
                    optimizer.zero_grad()                                             # clear previous gradients
                    logits_ = model(x_batch)                                          # forward pass
                    loss = criterion(logits_, y_batch)                                # computes mean batch loss
                    loss.backward()                                                   # backward pass: compute gradients
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
                    optimizer.step()                                                  # update model weights
                ###################################################    
                bsize = y_batch.size(0)
                total_loss += loss.item() * bsize                                 # Total loss contribution of this batch
                
                predictions = logits_.argmax(dim = -1)                            # gets the predicted class for each window: picks the highest score in the last dimension (one highest score per window among the 4 classes)
                total_correct += (predictions == y_batch).sum().item()            # number of correct predictions
                total_samples += bsize
                loop.set_postfix(loss= f"{total_loss/total_samples:.4f}", acc=f"{total_correct/total_samples:.4f}")
    
            train_loss, train_acc = total_loss / total_samples, total_correct / total_samples
            val_loss, val_acc, report = validate_model(model, val_loader, device, criterion)
            if device.type == "cuda":
                torch.cuda.synchronize()
            epoch_time = time.time() - epoch_start # END EPOCH TIME
            epoch_history.append({
                "epoch": epoch + 1,
                "tr_loss": float(train_loss),
                "tr_acc": float(train_acc),
                "val_loss": float(val_loss),
                "val_acc": float(val_acc)                
            })
            print(f"\nEpoch: {epoch+1}/{num_epochs} | tr_Loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s")
            if epoch == 0 and prof_out is not None:
                print("--- CUDA PROFILE --- (epoch 1, batch 2)")
                print(prof_out)    
            log_file.write(f"Epoch: {epoch+1}/{num_epochs} | tr_loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s\n")
            log_file.flush()
                
            if val_loss < best_val_loss - 1e-12:
                best_val_loss = float(val_loss)
                best_val_acc = float(val_acc)
                best_epoch = epoch + 1                   
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping at epoch {epoch + 1} | Best Validation Loss: {best_val_loss:.4f}")
                    break
                    
        # Load Best Model !            
        if best_state is not None:
            model.load_state_dict(best_state)
            torch.save(best_state, model_name)
            # Run val on best model to generate report
            _, _, report = validate_model(model, val_loader, device, criterion) 
            print(report)
            log_file.write(f"\nBest Model Validation Report: \n{str(report)}")        
        log_file.write(f"TRAINING ENDING AT: {datetime.now()}\n")                           
        
        #  ----------- TEST -----------  
        # -- Reset Peak Stats --
        if device.type == "cuda":
            peak_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)
            print(f"Peak GPU memory allocatated: {peak_mem_gb:.2f}GB\n")
        # ----------------------        
        acc, report, f1, conf_matrix = test_model(model, test_loader, device)
        seed_result = {
            "seed": int(seed),
            "fold": int(1),
            "history": epoch_history,
            "summary": {
                "best_epoch": best_epoch,
                "best_val_loss": float(best_val_loss),
                "best_val_acc": float(best_val_acc),
                "test_accuracy": float(acc),
                "test_report": report,
                "test_f1": float(f1),
                "test_conf_matrix": conf_matrix.tolist()               
            }
        }
        save_json("OPP", 4, seed, seed_result)
        print(f"{'-'*90}")
        print(f"Test Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
        log_file.write(f"\nTest Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
#  ----------------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Training set class distribution:
{'STAND': 4652, 'WALK': 2791, 'SIT': 2704, 'LIE': 528}
------------------------------------------------------------------------------------------
Saved as optimizer_assignment_table.txt


Epoch 1/30:   5%|▍         | 4/84 [00:01<00:39,  2.01it/s, acc=0.6484, loss=0.8016]


KeyboardInterrupt: 